# R-GCN Benchmark em Colab

Notebook prático para correr o benchmark `rgcn_benchmark` em Google Colab. O Colab é a escolha recomendada aqui porque normalmente dá acesso mais simples a NVIDIA CUDA. Se preferires Kaggle, a maior parte dos passos também funciona, mas a configuração de GPU e instalação de pacotes tende a ser menos direta.


## 1. Escolher ambiente

**Recomendação:** usa **Colab** para este projeto. Se precisares de persistência local ou já estiveres a trabalhar dentro de um dataset, adapta os caminhos para Kaggle.

A diferença prática é esta:
- Colab: GPU NVIDIA costuma estar mais fácil de ativar.
- Kaggle: também pode funcionar, mas a instalação de dependências e acesso ao repositório costuma ser menos flexível.


## 2. Configurar runtime no Colab

No Colab, abre `Runtime > Change runtime type` e escolhe **GPU**.

Depois, corre a célula seguinte para confirmar que o runtime tem CUDA disponível.


In [ ]:
import os
import sys
import subprocess
import shutil

try:
    import torch
    print('torch version:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    print('cuda version:', torch.version.cuda)
    print('device count:', torch.cuda.device_count())
    if torch.cuda.is_available():
        print('gpu name:', torch.cuda.get_device_name(0))
    else:
        print('No CUDA GPU detected. In Colab, go to Runtime > Change runtime type and select GPU.')
except Exception as exc:
    print('Torch check failed:', exc)

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)
else:
    print('nvidia-smi not found in this runtime.')

## 3. Instalar dependências via pip

A instalação direta de `requirements.txt` pode falhar no Colab por causa dos pins de `torch`/`dgl`. Por isso, a célula seguinte instala as dependências de forma mais segura, mantendo o `torch` do Colab e escolhendo um wheel de DGL compatível quando possível.


In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/GoncaloGarranha/drug-recommender-system.git')
REPO_DIR = Path('/content/drug-recommender-system')
REQ = REPO_DIR / 'requirements.txt'

if REPO_DIR.exists() and not REQ.exists():
    print('repository exists but requirements.txt is missing; removing incomplete clone and retrying...')
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    print('cloning repository first because requirements.txt lives inside it...')
    clone_result = subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], text=True, capture_output=True)
    print(clone_result.stdout)
    print(clone_result.stderr)
    if clone_result.returncode != 0:
        raise RuntimeError(f'git clone failed with exit code {clone_result.returncode}. Check REPO_URL and network access.')
else:
    print('repository already present:', REPO_DIR)

if not REQ.exists():
    print('Available files in repo root:', [p.name for p in REPO_DIR.iterdir()][:20])
    raise FileNotFoundError(f'requirements.txt still not found at {REQ}. Clone the correct repository or upload the repo manually.')

print('upgrading pip...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=False)

print('installing pure-python dependencies from requirements.txt, excluding torch/dgl pins...')
filtered_req = Path('/content/requirements-colab.txt')
if REQ.exists():
    lines = []
    for line in REQ.read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith('torch==') or stripped.startswith('dgl==') or stripped.startswith('torchdata=='):
            continue
        if stripped:
            lines.append(stripped)
    filtered_req.write_text('\n'.join(lines) + '\n')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(filtered_req)], check=False)
else:
    print('requirements.txt still not found; check clone URL or upload the repo manually.')

# Try to install a GPU-capable DGL wheel compatible with the current Colab torch.
# If none matches, fall back to a CPU DGL wheel so the notebook still runs.
try:
    import torch
    torch_minor = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
    print('detected torch:', torch.__version__, 'cuda:', torch.version.cuda)
except Exception as exc:
    torch_minor = None
    print('could not detect torch version:', exc)

candidate_urls = []
if torch_minor:
    for minor in [torch_minor, '2.6', '2.5', '2.4', '2.3']:
        candidate_urls.append(f'https://data.dgl.ai/wheels/torch-{minor}/cu121/repo.html')

installed = False
for url in candidate_urls:
    print('trying DGL wheel index:', url)
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dgl', '-f', url])
    if result.returncode == 0:
        installed = True
        print('installed DGL from:', url)
        break

if not installed:
    print('GPU wheel not found; installing CPU DGL fallback')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dgl==1.1.2'], check=False)


## 4. Montar Google Drive

Se quiseres guardar checkpoints e métricas no Drive, monta-o agora. Se estiveres no Kaggle, troca esta secção por caminhos locais do ambiente.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
MODEL_OUT = DRIVE_ROOT / 'rgcn_benchmark_runs'
MODEL_OUT.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)
print('Output dir:', MODEL_OUT)


## 5. Fazer upload de dados / usar Kaggle datasets

Se não tens os ficheiros já no Drive, usa upload manual ou a API do Kaggle para trazer os datasets.


In [ ]:
from google.colab import files

# Upload manual de ficheiros pequenos, se precisares.
# Exemplo: um CSV ou um zip com dados auxiliares.
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

# Opcional: Kaggle API (descomenta e ajusta o slug do dataset).
# !pip install -q kaggle
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d owner/dataset -p /content/data
# !unzip -o /content/data/dataset.zip -d /content/data


## 6. Clonar repositório e carregar código

Se o repositório já estiver no Drive, salta o clone e aponta `REPO_URL` para a tua cópia local. Caso contrário, usa o clone direto.


In [ ]:
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/GoncaloGarranha/drug-recommender-system.git')
REPO_DIR = Path('/content/drug-recommender-system')

if not REPO_DIR.exists():
    print('cloning repository:', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=False)
else:
    print('repository already exists at:', REPO_DIR)

%cd /content/drug-recommender-system

if (REPO_DIR / 'setup.py').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=False)
else:
    print('No setup.py at repo root; using sys.path only.')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('cwd:', Path.cwd())


## 7. Executar scripts / treino / inferência

Para um teste rápido, corre primeiro um `smoke run` com poucas iterações. Depois aumenta para o treino completo quando confirmares que tudo está estável.


In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = Path('models/rgcn_benchmark/configs/wn18.yaml')
SMOKE_CONFIG = Path('/content/wn18-colab-smoke.yaml')
SMOKE_TEST = True

if SMOKE_TEST:
    cfg = yaml.safe_load(CONFIG_PATH.read_text())
    cfg['training']['max_iterations'] = 50
    cfg['training']['eval_every'] = 25
    cfg['training']['burnin_iterations'] = 25
    SMOKE_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
    config_to_run = SMOKE_CONFIG
else:
    config_to_run = CONFIG_PATH

print('config_to_run:', config_to_run)

# Smoke run
subprocess.run([sys.executable, '-m', 'models.rgcn_benchmark.src.train', '--config', str(config_to_run)], check=False)


In [ ]:
# Evaluation on the best checkpoint (test split)
subprocess.run([
    sys.executable,
    '-m',
    'models.rgcn_benchmark.src.evaluate',
    '--runs_dir',
    'models/rgcn_benchmark/runs/wn18',
    '--split',
    'test',
], check=False)


## 8. Salvar modelos e outputs no Drive

Depois do treino, copia a pasta de runs para o Drive para não perder checkpoints e métricas.


In [ ]:
import shutil

run_dir = Path('models/rgcn_benchmark/runs/wn18')
drive_target = MODEL_OUT / 'wn18'
drive_target.mkdir(parents=True, exist_ok=True)

if run_dir.exists():
    shutil.copytree(run_dir, drive_target, dirs_exist_ok=True)
    print('Saved run to:', drive_target)
else:
    print('Run directory not found yet:', run_dir)

# Example for direct model save
# import torch
# torch.save(model.state_dict(), '/content/drive/MyDrive/models/model.pt')


## 9. Visualizar logs, outputs e usar output pane

No Colab, o output das células é a forma mais simples de acompanhar o treino. Para logs, escreve para ficheiros e imprime métricas a cada avaliação.


In [ ]:
from pathlib import Path
import json

latest_run = None
runs_root = Path('models/rgcn_benchmark/runs')
if runs_root.exists():
    candidates = [p for p in runs_root.iterdir() if p.is_dir()]
    if candidates:
        latest_run = max(candidates, key=lambda p: p.stat().st_mtime)
        print('latest run:', latest_run)
        best_metrics = latest_run / 'best_metrics.json'
        if best_metrics.exists():
            print(json.loads(best_metrics.read_text()))
        train_log = latest_run / 'train_log.json'
        if train_log.exists():
            log_data = json.loads(train_log.read_text())
            print('last log row:', log_data[-1] if log_data else 'empty')
else:
    print('No runs found yet.')

# Example of watching a log file locally if you are in a Linux shell:
# !tail -f models/rgcn_benchmark/runs/wn18/train_log.json &


## 10. Testes rápidos e verificação de integridade

Antes de um treino longo, corre sempre uma verificação mínima de imports, dependências e shapes.


In [ ]:
import importlib.util
import torch

modules = ['torch', 'dgl', 'yaml', 'pandas', 'numpy']
for name in modules:
    print(name, '->', importlib.util.find_spec(name) is not None)

print('cuda available:', torch.cuda.is_available())

# Minimal sanity imports for the project
import models.rgcn_benchmark.src.train as train_mod
import models.rgcn_benchmark.src.evaluate as eval_mod
print('project imports: ok')

# If you have tests in the repo, you can uncomment:
# !pytest -q tests/
# Or run a targeted script:
# !python scripts/sanity_check.py
